## API de modelo Qwen 2.5-7B
Acceso desde collab al modelo Qwen 2.5-7B para tutor de programación.

In [ ]:
!pip install bitsandbytes pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.2 MB/s eta 0:00:00


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import re
import time
import unicodedata
import logging
import torch
import threading
from enum import Enum
from pyngrok import ngrok
from typing import List, Dict, Any
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
#  CONFIGURACIÓN INICIAL
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True
)

app = FastAPI(title="Tutor IA - Qwen 2.5 7B", version="3.0.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

BASE_MODEL      = "Qwen/Qwen2.5-7B-Instruct"
MAX_CTX_TOKENS  = 8192
MAX_NEW_TOKENS  = 2048

In [ ]:
#  CARGA DEL MODELO  (T4 · 15 GB VRAM · 4-bit NF4)
logging.info(f"Cargando modelo {BASE_MODEL} en 4-bit NF4…")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)

model.eval()
torch.set_grad_enabled(False)
logging.info("✅ Modelo 7B cargado. Listo para inferencia.")

2026-06-11 19:36:57 | INFO     | Cargando modelo Qwen/Qwen2.5-7B-Instruct en 4-bit NF4…
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
2026-06-11 19:36:58 | INFO     | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-11 19:36:58 | INFO     | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"
2026-06-11 19:36:58 | INFO     | HTTP 

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

2026-06-11 19:36:59 | INFO     | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-11 19:36:59 | INFO     | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-11 19:36:59 | INFO     | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

2026-06-11 19:36:59 | INFO     | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-06-11 19:36:59 | INFO     | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-06-11 19:36:59 | INFO     | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
2026-06-11 19:36:59 | WARNING  | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-11 19:36:59 | INFO     | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/vocab.json "HTTP/1.1 200 OK"
2026-06-11 19:36:59 | INFO     | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.

...

2026-06-11 19:37:16 | INFO     | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/model-00001-of-00004.safetensors "HTTP/1.1 302 Found"
2026-06-11 19:37:16 | INFO     | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/model-00002-of-00004.safetensors "HTTP/1.1 302 Found"
2026-06-11 19:37:16 | INFO     | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/model-00004-of-00004.safetensors "HTTP/1.1 302 Found"
2026-06-11 19:37:16 | INFO     | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/model-00003-of-00004.safetensors "HTTP/1.1 302 Found"
2026-06-11 19:37:16 | INFO     | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/xet-read-token/a09a35458c702b33eeacc393d103063234e8bc28 "HTTP/1.1 200 OK"
2026

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
2026-06-11 19:49:19 | INFO     | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-11 19:49:19 | INFO     | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json "HTTP/1.1 200 OK"
2026-06-11 19:49:20 | INFO     | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

2026-06-11 19:49:20 | INFO     | ✅ Modelo 7B cargado. Listo para inferencia.


In [ ]:
#  UTILIDADES DE TEXTO
def normalizar(texto: str) -> str:
    """Elimina acentos y convierte a minúsculas para comparaciones robustas."""
    return "".join(
        c for c in unicodedata.normalize("NFD", texto.lower())
        if unicodedata.category(c) != "Mn"
    )

def limpiar_respuesta(texto: str) -> str:
    """Elimina artefactos, etiquetas de pensamiento y basura de Markdown."""
    # 1. Elimina el bloque de razonamiento interno
    texto = re.sub(r'<razonamiento>.*?</razonamiento>', '', texto, flags=re.DOTALL)

    # 2. Elimina tokens especiales del chat
    texto = re.sub(r"<\|im_end\|>.*", "", texto, flags=re.DOTALL)
    texto = re.sub(r"<\|im_start\|>.*", "", texto, flags=re.DOTALL)

    texto = texto.strip()

    # 3. Limpieza agresiva del bloque markdown global y la palabra 'undefined'
    # Quita el ```markdown o ``` al inicio
    texto = re.sub(r"^```[a-zA-Z]*\n", "", texto)
    # Quita el ``` del final y cualquier basura (como undefined) que le siga
    texto = re.sub(r"\n```\s*[a-zA-Z]*\s*$", "", texto)

    return texto.strip()

def calcular_nivel_frustracion(mensajes: list) -> float:
    """Retorna un score de 0.0 a 1.0 basado en señales de frustración acumuladas."""
    score = 0.0
    mensajes_usuario = [m for m in mensajes if m.get("role") == "user"]

    for i, msg in enumerate(mensajes_usuario[-5:]):
        texto = normalizar(msg.get("content", ""))
        peso = (i + 1) / 5

        if re.search(r'\b(no funciona|sigue sin|otra vez|de nuevo|error|falla|no corre)\b', texto):
            score += 0.2 * peso
        if re.search(r'\b(no entiendo|no se|confundido|ayuda|me rindo|harto|odio esto)\b', texto):
            score += 0.25 * peso
        if len(texto) < 15 and "?" in texto:
            score += 0.15 * peso

    if len(mensajes_usuario) > 5:
        score += 0.1

    return min(score, 1.0)

In [ ]:
class AppState(str, Enum):
    TUTOR_BASE = "TUTOR_BASE"
    RESTRINGIDO = "RESTRINGIDO"
    DEBUGGING = "DEBUGGING"
    COMPETITIVO = "COMPETITIVO"
    EMPATIA = "EMPATIA"
    VISUALIZACION = "VISUALIZACION"
    REVISION = "REVISION"

In [ ]:
#  MÁQUINA DE ESTADOS FINITOS (FSM)
def transicion_estado(mensaje: str, estado_anterior: str, lock_competitivo: bool = False, frustracion: float = 0.0) -> str:
    if lock_competitivo or estado_anterior == AppState.COMPETITIVO:
        if re.search(r'\b(normal|salir|terminar|finalizar)\b', mensaje.lower()):
            return AppState.TUTOR_BASE
        return AppState.COMPETITIVO

    if frustracion > 0.6:
        return AppState.EMPATIA

    msg = normalizar(mensaje)

    # 1. PRIORIDAD ALTA: VISUALIZACION
    if re.search(r'\b(anim\w*|visualiz\w*|gif|dibuj\w*|grafic\w*|mue\w*me|ver el algoritmo)\b', msg):
        return AppState.VISUALIZACION

    # 2. REVISION
    if re.search(r'\b(revis\w*|mejorar\w*|optimiz\w*|buenas practica\w*|pep8|clean code|refactor\w*)', msg):
        return AppState.REVISION

    # 3. DEBUGGING
    if re.search(r'\b(error|bug\w*|fall\w*|no funcion\w*|excepcion|traceback|syntax|sintaxis|congelado|infinit\w*|nul\w*)', msg):
        return AppState.DEBUGGING

    # 4. RESTRINGIDO
    if re.search(r'\b(haz|resuelv\w*|dame|pasame|hazlo|escribeme|dame el codigo|solucion\w*)', msg):
        return AppState.RESTRINGIDO

    # 5. COMPETITIVO
    if re.search(r'\b(leetcode|codeforces|competitiv\w*|icpc|big o|complejidad|o grande|estructura de datos)', msg):
        return AppState.COMPETITIVO

    return AppState.TUTOR_BASE

In [ ]:
#  SYSTEM PROMPTS POR ESTADO PEDAGÓGICO
def obtener_system_prompt(estado: str, frustracion: float = 0.0) -> str:
    base_prompt = """Eres un Tutor Socrático de Programación de nivel universitario experto.
Tu objetivo es que el alumno entienda profundamente los conceptos y desarrolle su lógica, NO que copie y pegue.

REGLAS DE ORO:
1. EXPLICACIÓN ESTRUCTURADA: Usa Markdown (títulos, viñetas, negritas) para claridad.
2. CÓDIGO PARCIAL (ESQUELETOS): NUNCA entregues el algoritmo central resuelto ni bucles con la lógica final. SÍ proporciona esqueletos, firmas de funciones y comentarios `# TODO: tu lógica aquí`.
3. PREGUNTA SOCRÁTICA: Termina SIEMPRE tu respuesta con UNA (1) pregunta reflexiva específica sobre el siguiente paso que el alumno debe codificar.
4. PENSAMIENTO OCULTO: ANTES de responder, usa las etiquetas <razonamiento> y </razonamiento> para planear tu explicación didáctica."""

    modificadores = {
        AppState.TUTOR_BASE: (
            "El alumno quiere aprender un concepto. PASOS:\n"
            "1. Explica la teoría con una analogía creativa de la vida real.\n"
            "2. Muestra la estructura (esqueleto) del código.\n"
            "3. Haz la pregunta socrática final."
        ),
        AppState.RESTRINGIDO: (
            "ALERTA: El usuario pide la solución directa. PASOS:\n"
            "1. Rechaza la petición con firmeza pero amabilidad: 'Mi trabajo es guiarte, no hacerlo por ti'.\n"
            "2. Explica el concepto general.\n"
            "3. Entrega un esqueleto vacío y pregunta cómo implementaría el primer paso."
        ),
        AppState.DEBUGGING: (
            "El usuario tiene un bug. PASOS:\n"
            "1. Identifica el error y explica el ORIGEN CONCEPTUAL (memoria, sintaxis, lógica).\n"
            "2. NO reescribas la línea corregida. Explica qué debe cambiar y por qué.\n"
            "3. Pregunta: '¿Cómo modificarías tu bucle/condición basándote en esto?'"
        ),
        AppState.COMPETITIVO: (
            "MODO JUEZ AUTOMÁTICO (ICPC/Codeforces). PASOS:\n"
            "1. Inicia ESTRICTAMENTE con un veredicto en negritas: **[AC]**, **[WA]**, **[TLE]** o **[CE]**.\n"
            "2. Si es WA/TLE, da un análisis de complejidad Big O o un caso borde exacto donde falla.\n"
            "3. NO des la solución. Haz una pregunta críptica sobre el cuello de botella."
        ),
        AppState.EMPATIA: (
            "El alumno está frustrado. PASOS:\n"
            "1. Valida su emoción brevemente: 'Es completamente normal atascarse aquí, a todos nos pasa'.\n"
            "2. Ignora el problema grande. Aísla un micro-paso ridículamente sencillo.\n"
            "3. Pide que solo resuelva ese micro-paso para recuperar la confianza."
        ),
        AppState.VISUALIZACION: (
            "Genera código Python COMPLETO usando la clase Visualizer.\n"
            "REGLAS:\n"
            "1. `from visualizer import Visualizer` y `viz = Visualizer()`\n"
            "2. Elige el método según el algoritmo:\n"
            "   - Sorting/búsqueda en arreglo → viz.capture(arr, titulo, highlight=[i,j])\n"
            "   - Programación dinámica      → viz.capture_matrix(dp, highlight_cell=(i,j))\n"
            "   - DFS/BFS/Dijkstra           → viz.capture_graph(nodes, edges, visited, current)\n"
            "   - Árboles binarios           → viz.capture_tree(root_as_list)\n"
            "   - Listas enlazadas           → viz.capture_linked_list(values, current_idx)\n"
            "3. viz.capture() va DENTRO del bucle principal, no fuera\n"
            "4. Terminar SIEMPRE con viz.save_animation()\n"
            "5. Código 100% funcional, sin TODOs\n"
        ),
        AppState.REVISION: (
            "El alumno pide revisar su código funcional. PASOS:\n"
            "1. Elogia brevemente que funcione.\n"
            "2. Señala 1 o 2 áreas de mejora (nombres de variables, complejidad, legibilidad).\n"
            "3. Muestra un pequeño fragmento de cómo se vería esa mejora específica y pregunta si quiere aplicarla."
        )
    }

    prompt_final = base_prompt + "\n\nMODO ACTUAL:\n" + modificadores.get(estado, modificadores[AppState.TUTOR_BASE])

    if frustracion > 0.6 and estado != AppState.COMPETITIVO:
        prompt_final += "\n\n⚠️ INSTRUCCIÓN CRÍTICA: El alumno registra alto estrés. Usa un tono de mentor extremadamente paciente, cálido y detallado. Reduce la carga cognitiva al mínimo."

    return prompt_final

In [ ]:
#  INTERCEPTOR ANTI-TRAMPAS
def es_respuesta_insegura(texto: str) -> bool:
    """Detecta si el modelo generó un bloque de código funcional completo en modo RESTRINGIDO."""
    patron = r"```(?:python|javascript|java|cpp|c|js)?\s*\n(.*?)\n```"
    bloques = re.findall(patron, texto, re.DOTALL | re.IGNORECASE)

    for bloque in bloques:
        lineas_reales = [
            l.strip() for l in bloque.split("\n")
            if l.strip()
            and not l.strip().startswith("#")
            and l.strip() != "pass"
            and "..." not in l
            and "todo" not in l.lower()
        ]
        if len(lineas_reales) >= 5:
            return True
    return False

RESPUESTA_BLOQUEO = (
    "🛡️ **Alto ahí.** Como tu tutor, mi rol es guiarte para que **tú** construyas la solución, no hacerlo por ti.\n\n"
    "Empecemos desde lo básico:\n"
    "```python\n"
    "def resolver_problema(datos):\n"
    "    # TODO: ¿Cuál es el primer dato que necesitas procesar o validar?\n"
    "    pass\n"
    "```\n"
    "¿Qué crees que debería ir en ese primer paso?"
)

#  GESTIÓN DE CONTEXTO
def truncar_mensajes(
    mensajes: list, tokenizer, max_tokens: int, margen: int = 300
) -> list:
    """
    Conserva siempre:
      - El system prompt (índice 0)
      - El último mensaje del usuario (índice -1)
    Luego agrega mensajes intermedios desde el más reciente hacia atrás
    hasta llenar el presupuesto de tokens.
    """
    sys_msg  = mensajes[0]
    ultimo   = mensajes[-1]
    intermedios = mensajes[1:-1]

    tokens_base = len(tokenizer.encode(
        str(sys_msg["content"]) + str(ultimo["content"]),
        add_special_tokens=False,
    ))

    seleccionados = []
    tokens_usados = tokens_base

    for msg in reversed(intermedios):
        t = len(tokenizer.encode(str(msg["content"]), add_special_tokens=False))
        if tokens_usados + t < max_tokens - margen:
            seleccionados.insert(0, msg)
            tokens_usados += t
        else:
            break  # Dejamos de agregar; los más antiguos se descartan

    logging.info(
        f"[CONTEXTO] Mensajes conservados: {1 + len(seleccionados) + 1} "
        f"| Tokens estimados: {tokens_usados}"
    )
    return [sys_msg] + seleccionados + [ultimo]

In [ ]:
# MODELO DE DATOS
class ChatRequest(BaseModel):
    messages:        List[Dict[str, Any]] = Field(..., min_length=1)
    is_competitive_locked: bool = Field(False)
    estado_anterior: str  = Field(AppState.TUTOR_BASE)

# ENDPOINT PRINCIPAL
@app.post("/generate")
async def generate_response(request: ChatRequest):
    start_time = time.time()
    try:
        ultimo_msg = request.messages[-1].get("content", "")

        # 1. Calcular frustración REAL basada en el historial
        nivel_frustracion = calcular_nivel_frustracion(request.messages)

        # 2. Procesar la FSM
        estado = transicion_estado(
            mensaje=ultimo_msg,
            estado_anterior=request.estado_anterior,
            lock_competitivo=request.is_competitive_locked,
            frustracion=nivel_frustracion
        )

        sys_prompt = obtener_system_prompt(estado, nivel_frustracion)
        logging.info(f"[ESTADO FSM] Transición: {request.estado_anterior} -> {estado} | Frustración: {nivel_frustracion:.2f}")

        # 3. Construir lista de mensajes
        mensajes = [{"role": "system", "content": sys_prompt}] + list(request.messages)

        # 4. REFUERZO DE COLA (Tail Reinforcement) como mensaje de sistema adicional
        if len(mensajes) > 1:
            instruccion_fantasma = ""
            if estado == AppState.COMPETITIVO:
                instruccion_fantasma = "\n[SYSTEM: Modo Juez. Evalúa y da veredicto (AC, WA, TLE, MLE, CE). NO corrijas el código, solo señala fallos de complejidad o casos límite. Si es WA: proporciona input → expected → actual. Si es TLE: calcula la complejidad Big O actual.]"
            elif estado == AppState.RESTRINGIDO:
                instruccion_fantasma = "\n[SYSTEM: El usuario intenta que le des el código final. RECHAZA rotundamente entregar la lógica completa. Solo entrega esqueletos vacíos con 'TODOs'.]"
            elif estado == AppState.DEBUGGING:
                instruccion_fantasma = "\n[SYSTEM: Explica el bug a nivel conceptual. NO le des el código ya arreglado.]"
            elif estado == AppState.VISUALIZACION:
                instruccion_fantasma = "\n[SYSTEM: CHECKLIST antes de terminar tu respuesta: ¿Importaste Visualizer correctamente?, ¿viz.capture() está DENTRO del bucle principal (no fuera)?, ¿El algoritmo tiene datos de ejemplo hardcodeados (sin input())?, ¿La última línea es viz.save_animation()?, ¿Hay algún TODO pendiente? → Si sí, COMPLÉTALO antes de responder, Código con TODOs no se puede ejecutar.]"

            if instruccion_fantasma:
                # Lo añadimos como un mensaje de sistema justo antes de la generación para máxima atención del modelo
                mensajes.append({"role": "system", "content": instruccion_fantasma})

        # 5. Aplicar chat template de Qwen
        text = tokenizer.apply_chat_template(
            mensajes, tokenize=False, add_generation_prompt=True
        )

        # 6. Validar y truncar contexto
        ctx_tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(ctx_tokens) > MAX_CTX_TOKENS:
            logging.warning(f"Contexto excedido ({len(ctx_tokens)} tokens). Truncando…")
            mensajes_truncados = truncar_mensajes(mensajes, tokenizer, MAX_CTX_TOKENS)
            # Reaplicar template tras truncar
            text = tokenizer.apply_chat_template(
                mensajes_truncados, tokenize=False, add_generation_prompt=True
            )

        inputs = tokenizer([text], return_tensors="pt").to(model.device)

        # 7. Configuración dinámica de generación
        temp_map = {
            AppState.COMPETITIVO:   0.1,
            AppState.VISUALIZACION: 0.2,
            AppState.RESTRINGIDO:   0.25,
            AppState.DEBUGGING:     0.35,
            AppState.REVISION:      0.4,
            AppState.TUTOR_BASE:    0.45,
            AppState.EMPATIA:       0.5,
        }
        temperatura_actual = temp_map.get(estado, 0.3)

        # 8. Generación
        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=temperatura_actual,
                top_p=0.92,
                top_k=50,
                repetition_penalty=1.15,
                do_sample=True,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        # 9. Decodificar y Limpiar
        nuevos_ids = gen_ids[:, inputs.input_ids.shape[1]:]
        res = tokenizer.batch_decode(nuevos_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

        # Limpieza de artefactos de Qwen
        res = re.sub(r'<razonamiento>.*?</razonamiento>', '', res, flags=re.DOTALL)
        res = re.sub(r'<\|im_end\|>.*', '', res, flags=re.DOTALL).strip()

        # 10. Interceptor de Seguridad
        if estado == AppState.RESTRINGIDO and es_respuesta_insegura(res):
            logging.warning("[SEGURIDAD] Fuga de código bloqueada.")
            res = RESPUESTA_BLOQUEO

        duration = round(time.time() - start_time, 2)
        logging.info(f"[OK] Tiempo: {duration}s | Estado: {estado} | Tokens: {len(ctx_tokens)}")

        return {
            "response": res,
            "estado_detectado": estado,
            "nivel_frustracion": round(nivel_frustracion, 2)
        }

    except torch.cuda.OutOfMemoryError:
        logging.error("OOM: VRAM insuficiente. Liberando caché…")
        torch.cuda.empty_cache()
        return {"response": "La memoria de la GPU se agotó. Intenta acortar el historial o reiniciar el chat.", "estado_detectado": "ERROR_VRAM"}
    except Exception as e:
        logging.error(f"[ERROR] Fallo inesperado: {e}", exc_info=True)
        return {"response": "Ocurrió un error interno. Por favor intenta de nuevo.", "estado_detectado": "ERROR"}


#  HEALTH CHECK
@app.get("/health")
async def health():
    vram_libre = (
        round(torch.cuda.mem_get_info()[0] / 1e9, 2)
        if torch.cuda.is_available()
        else None
    )
    return {"status": "ok", "vram_libre_gb": vram_libre}


#  ARRANQUE
if __name__ == "__main__":
    import uvicorn
    from google.colab import userdata

    try:
        ngrok_token = userdata.get('NGROK_AUTH_TOKEN')
        ngrok.set_auth_token(ngrok_token)
    except Exception:
        print("ADVERTENCIA: No se encontró el secreto 'NGROK_AUTH_TOKEN' en Colab.")

    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
    print("=" * 60)
    print(f"IA_URL={public_url}/generate")
    print("=" * 60)

    def run_server():
        logging.info("Iniciando servidor FastAPI en puerto 8000…")
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info", use_colors=True, reload=False)

    thread = threading.Thread(target=run_server, daemon=True)
    thread.start()

    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\nServidor detenido manualmente.")

2026-06-11 20:40:40 | INFO     | Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
2026-06-11 20:40:40 | INFO     | Opening tunnel named: http-8000-c03432c7-015d-4d67-bac1-5e1a355f78ca
2026-06-11 20:40:40 | INFO     | t=2026-06-11T20:40:40+0000 lvl=info msg="no configuration paths supplied"
2026-06-11 20:40:40 | INFO     | t=2026-06-11T20:40:40+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
2026-06-11 20:40:40 | INFO     | t=2026-06-11T20:40:40+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
2026-06-11 20:40:40 | INFO     | t=2026-06-11T20:40:40+0000 lvl=info msg="FIPS 140 mode" enabled=false
2026-06-11 20:40:40 | INFO     | t=2026-06-11T20:40:40+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
2026-06-11 20:40:40 | INFO     | t=2026-06-11T20:40:40+0000 lvl=info msg="client session established" obj=tunnels.session
2026-06-11 20:40:4

IA_URL=https://a0f0-136-109-69-29.ngrok-free.app/generate


INFO:     Started server process [10121]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
2026-06-11 20:41:22 | INFO     | t=2026-06-11T20:41:22+0000 lvl=info msg="join connections" obj=join id=2857706bdaf4 l=127.0.0.1:8000 r=[2806:2f0:a780:4c5:e214:e04f:5c13:d750]:54162
2026-06-11 20:41:22 | INFO     | [ESTADO FSM] Transición: TUTOR_BASE -> AppState.TUTOR_BASE | Frustración: 0.00
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
2026-06-11 20:42:51 | INFO     | [OK] Tiempo: 88.82s | Estado: AppState.TUTOR_BASE | Tokens: 344


INFO:     2806:2f0:a780:4c5:e214:e04f:5c13:d750:0 - "POST /generate HTTP/1.1" 200 OK


2026-06-11 20:44:32 | INFO     | t=2026-06-11T20:44:32+0000 lvl=info msg="join connections" obj=join id=28f73871ddde l=127.0.0.1:8000 r=[2806:2f0:a780:4c5:e214:e04f:5c13:d750]:52782
2026-06-11 20:44:32 | INFO     | [ESTADO FSM] Transición: TUTOR_BASE -> AppState.TUTOR_BASE | Frustración: 0.00
2026-06-11 20:45:49 | INFO     | [OK] Tiempo: 77.5s | Estado: AppState.TUTOR_BASE | Tokens: 1329


...

2026-06-11 20:56:59 | INFO     | [ESTADO FSM] Transición: COMPETITIVO -> AppState.COMPETITIVO | Frustración: 0.00
2026-06-11 20:56:59 | INFO     | t=2026-06-11T20:56:59+0000 lvl=info msg="join connections" obj=join id=430e7ae9873d l=127.0.0.1:8000 r=[2806:2f0:a780:4c5:e214:e04f:5c13:d750]:63530
2026-06-11 20:57:17 | INFO     | [OK] Tiempo: 18.25s | Estado: AppState.COMPETITIVO | Tokens: 895


INFO:     2806:2f0:a780:4c5:e214:e04f:5c13:d750:0 - "POST /generate HTTP/1.1" 200 OK


2026-06-11 20:58:09 | INFO     | [ESTADO FSM] Transición: COMPETITIVO -> AppState.COMPETITIVO | Frustración: 0.00
2026-06-11 20:58:09 | INFO     | t=2026-06-11T20:58:09+0000 lvl=info msg="join connections" obj=join id=0cf351087e5a l=127.0.0.1:8000 r=[2806:2f0:a780:4c5:e214:e04f:5c13:d750]:60421
2026-06-11 20:58:31 | INFO     | [OK] Tiempo: 21.95s | Estado: AppState.COMPETITIVO | Tokens: 1253


INFO:     2806:2f0:a780:4c5:e214:e04f:5c13:d750:0 - "POST /generate HTTP/1.1" 200 OK

Servidor detenido manualmente.


2026-06-11 21:05:08 | INFO     | t=2026-06-11T21:05:08+0000 lvl=info msg="received stop request" obj=app stopReq="{err:<nil> restart:false}"
2026-06-11 21:05:08 | INFO     | t=2026-06-11T21:05:08+0000 lvl=info msg="session closing" obj=tunnels.session err=nil
2026-06-11 21:05:08 | INFO     | t=2026-06-11T21:05:08+0000 lvl=info msg="accept failed" obj=tunnels.session obj=csess id=7e9a9b2235fd err="reconnecting session closed"


# Pruebas

In [ ]:
# Lista de casos de prueba: (mensaje_usuario, estado_esperado, estado_anterior_simulado)
casos_prueba = [
    # DEBUGGING (requieren palabras de error + contexto de código)
    ("Tengo un error en esta linea 5", AppState.DEBUGGING, "TUTOR_BASE"),
    ("mi codigo no funciona y me da traceback", AppState.DEBUGGING, "TUTOR_BASE"),
    ("por que mi variable es nula? def funcion():", AppState.DEBUGGING, "TUTOR_BASE"),
    ("ayuda con este bug, mi codigo es ```x = 1```", AppState.DEBUGGING, "TUTOR_BASE"),
    ("sintaxis incorrecta en print()", AppState.DEBUGGING, "TUTOR_BASE"),
    ("bucle infinito en mi clase main", AppState.DEBUGGING, "TUTOR_BASE"),

    # RESTRINGIDO (imperativos directos)
    ("hazme el codigo de bubble sort", AppState.RESTRINGIDO, "TUTOR_BASE"),
    ("pasame la solucion del examen", AppState.RESTRINGIDO, "TUTOR_BASE"),
    ("escribeme la funcion entera", AppState.RESTRINGIDO, "TUTOR_BASE"),
    ("dame el codigo de esa tarea", AppState.RESTRINGIDO, "TUTOR_BASE"),
    ("resuelvame este problema", AppState.RESTRINGIDO, "TUTOR_BASE"),

    # VISUALIZACION
    ("me gustaria visualizar el funcionamiento del metodo de ordenamiento burbuja", AppState.VISUALIZACION, "TUTOR_BASE"),
    ("puedes visualizar como funciona?", AppState.VISUALIZACION, "TUTOR_BASE"),
    ("dibuja el proceso de ordenamiento", AppState.VISUALIZACION, "TUTOR_BASE"),
    ("hazme un gif de la lista", AppState.VISUALIZACION, "TUTOR_BASE"),
    ("graficar el heap sort", AppState.VISUALIZACION, "TUTOR_BASE"),
    ("muéstrame como se mueven los nodos", AppState.VISUALIZACION, "TUTOR_BASE"),

    # REVISION
    ("revisa mi codigo y dime si esta bien", AppState.REVISION, "TUTOR_BASE"),
    ("puedes mejorar esto?", AppState.REVISION, "TUTOR_BASE"),
    ("optimizar este bucle", AppState.REVISION, "TUTOR_BASE"),
    ("aplica buenas practicas a mi script", AppState.REVISION, "TUTOR_BASE"),
    ("que te parece mi estilo pep8?", AppState.REVISION, "TUTOR_BASE"),
    ("refactoriza esta funcion", AppState.REVISION, "TUTOR_BASE"),

    # COMPETITIVO
    ("tengo un ejercicio de leetcode", AppState.COMPETITIVO, "TUTOR_BASE"),
    ("necesito ayuda para codeforces", AppState.COMPETITIVO, "TUTOR_BASE"),
    ("cual es la complejidad temporal de esto?", AppState.COMPETITIVO, "TUTOR_BASE"),
    ("problema de programacion competitiva", AppState.COMPETITIVO, "TUTOR_BASE"),
    ("dime la notacion big o de esta funcion", AppState.COMPETITIVO, "TUTOR_BASE"),

    # TUTOR_BASE (conversaciones genéricas)
    ("hola, como estas?", AppState.TUTOR_BASE, "TUTOR_BASE"),
    ("que es una lista enlazada?", AppState.TUTOR_BASE, "TUTOR_BASE"),
    ("explicame que es un puntero", AppState.TUTOR_BASE, "TUTOR_BASE"),
    ("como declaro un arreglo?", AppState.TUTOR_BASE, "TUTOR_BASE"),

    # LOCK COMPETITIVO (verificar retorno a base)
    ("normal", AppState.TUTOR_BASE, "COMPETITIVO"),
    ("terminar modo competitivo", AppState.TUTOR_BASE, "COMPETITIVO"),
]

# Agregar tests adicionales hasta 50...
print(f"{'MENSAJE':<40} | {'ESPERADO':<15} | {'RESULTADO':<15} | {'STATUS'}")
print("-" * 85)

for msg, esperado, ant in casos_prueba:
    resultado = transicion_estado(msg, ant)
    status = "OK" if resultado == esperado else "FALLO"
    print(f"{msg[:38]:<40} | {esperado:<15} | {resultado:<15} | {status}")

MENSAJE                                  | ESPERADO        | RESULTADO       | STATUS
-------------------------------------------------------------------------------------
Tengo un error en esta linea 5           | AppState.DEBUGGING | AppState.DEBUGGING | OK
mi codigo no funciona y me da tracebac   | AppState.DEBUGGING | AppState.DEBUGGING | OK
por que mi variable es nula? def funci   | AppState.DEBUGGING | AppState.DEBUGGING | OK
ayuda con este bug, mi codigo es ```x    | AppState.DEBUGGING | AppState.DEBUGGING | OK
sintaxis incorrecta en print()           | AppState.DEBUGGING | AppState.DEBUGGING | OK
bucle infinito en mi clase main          | AppState.DEBUGGING | AppState.DEBUGGING | OK
hazme el codigo de bubble sort           | AppState.RESTRINGIDO | AppState.RESTRINGIDO | OK
pasame la solucion del examen            | AppState.RESTRINGIDO | AppState.RESTRINGIDO | OK
escribeme la funcion entera              | AppState.RESTRINGIDO | AppState.RESTRINGIDO | OK
dame el codigo de esa ta